In [1]:
import pandas as pd
import numpy as np 

In [2]:
df = pd.read_csv('Production_Crops_Livestock_E_All_Data.csv')

C:\Users\Aditya Wadekar\AppData\Local\Temp\ipykernel_10080\3421301631.py:1: DtypeWarning: Columns (11,14,17,20,23,26,29,32,35,38,41,44,47,50,53,56,59,62,65,68,71,74,77,80,83,86,89,92,95,98,101,104,107,110,113,116,119,122,125,128,131,134,137,140,143,146,149,152,155,158,161,164,167,170,173,176,179,182,185,188,191,194,197) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Production_Crops_Livestock_E_All_Data.csv')


In [3]:
df.shape

(79711, 198)

In [4]:
df.head()

,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Unit,Y1961,...,Y2020N,Y2021,Y2021F,Y2021N,Y2022,Y2022F,Y2022N,Y2023,Y2023F,Y2023N
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,ha,0.0,...,NaN,36862.0,A,NaN,36462.0,A,NaN,37000.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5412,Yield,kg/ha,NaN,...,NaN,1743.2,A,NaN,1742.0,A,NaN,1810.8,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5510,Production,t,0.0,...,NaN,64256.0,A,NaN,63515.0,A,NaN,67000.0,A,NaN
3,2,'004,Afghanistan,711,'01654,"Anise, badian, coriander, cumin, caraway, fenn...",5312,Area harvested,ha,NaN,...,NaN,25357.0,E,NaN,25403.0,E,NaN,25439.0,E,NaN
4,2,'004,Afghanistan,711,'01654,"Anise, badian, coriander, cumin, caraway, fenn...",5412,Yield,kg/ha,NaN,...,NaN,705.0,E,NaN,704.0,E,NaN,704.0,E,NaN


In [5]:
df_production = df[df['Element'] == 'Production'].copy()

In [6]:
aggregates = [
    "World", "Africa", "Americas", "Asia", "Europe", "Oceania",
    "Western Europe", "Eastern Europe", "Southern Europe", "Northern Europe",
    "Northern Africa", "Sub-Saharan Africa", "Eastern Africa", "Middle Africa",
    "Northern America", "Central America", "Caribbean", "South America",
    "Central Asia", "Eastern Asia", "Southern Asia", "South-Eastern Asia",
    "Western Asia", "Australia and New Zealand", "Melanesia", "Micronesia", "Polynesia",
    "European Union (27)", "Least Developed Countries", "Land Locked Developing Countries",
    "Small Island Developing States", "Low Income Food Deficit Countries",
    "Net Food Importing Developing Countries","South-eastern Asia", "Western Africa", "Southern Africa", 
    "USSR", "Yugoslav SFR", "Czechoslovakia", "Ethiopia PDR", 
    "Serbia and Montenegro", "Sudan (former)", "Belgium-Luxembourg"

]
df_countries = df_production[~df_production["Area"].isin(aggregates)].copy()

encoding_fixes = {
    "TÃ¼rkiye": "Türkiye",
    "CÃ´te d'Ivoire": "Côte d'Ivoire",
    "RÃ©union": "Réunion"
}

# Drop the overarching macro-aggregate "China" to avoid double counting mainland/Taiwan
df_countries = df_countries[df_countries["Area"] != "China"]

df_countries["Area"] = df_countries["Area"].replace(encoding_fixes)

In [7]:
# Set max rows to None (which means unlimited)
pd.set_option('display.max_rows', None)
df_countries['Area'].value_counts()


Area
China, mainland                                         219
Spain                                                   206
Bulgaria                                                199
Italy                                                   198
Türkiye                                                 197
Mexico                                                  195
Greece                                                  193
Romania                                                 187
Hungary                                                 186
France                                                  185
United States of America                                183
Portugal                                                182
Poland                                                  179
Slovakia                                                179
Morocco                                                 177
Iran (Islamic Republic of)                              177
Czechia                            

In [8]:
columns_to_keep = [col for col in df_countries.columns if not (col.endswith('F') or col.endswith('N'))]
df_clean_wide = df_countries[columns_to_keep]

In [9]:
df_clean_wide.shape

(22925, 72)

In [10]:
df_clean_wide.columns

Index(['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)',
       'Item', 'Element Code', 'Element', 'Unit', 'Y1961', 'Y1962', 'Y1963',
       'Y1964', 'Y1965', 'Y1966', 'Y1967', 'Y1968', 'Y1969', 'Y1970', 'Y1971',
       'Y1972', 'Y1973', 'Y1974', 'Y1975', 'Y1976', 'Y1977', 'Y1978', 'Y1979',
       'Y1980', 'Y1981', 'Y1982', 'Y1983', 'Y1984', 'Y1985', 'Y1986', 'Y1987',
       'Y1988', 'Y1989', 'Y1990', 'Y1991', 'Y1992', 'Y1993', 'Y1994', 'Y1995',
       'Y1996', 'Y1997', 'Y1998', 'Y1999', 'Y2000', 'Y2001', 'Y2002', 'Y2003',
       'Y2004', 'Y2005', 'Y2006', 'Y2007', 'Y2008', 'Y2009', 'Y2010', 'Y2011',
       'Y2012', 'Y2013', 'Y2014', 'Y2015', 'Y2016', 'Y2017', 'Y2018', 'Y2019',
       'Y2020', 'Y2021', 'Y2022', 'Y2023'],
      dtype='object')

In [11]:
# 5. Reshape from Wide to Long format (Melting)
# This turns columns like [Y1961, Y1962, Y1963] into a single 'Year' column and a 'Value' column
id_vars = [
    "Area Code (M49)",
    "Area",
    "Item Code (CPC)",
    "Item",
    "Element Code",
    "Element",
    "Unit"
]
year_cols = [col for col in df_clean_wide.columns if col.startswith('Y')]

df_long = pd.melt(
    df_clean_wide,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name="Year",
    value_name="Production_Value"
)

# Clean up the 'Year' column (convert 'Y2021' string to integer 2021)
df_long["Year"] = df_long["Year"].str.replace("Y", "").astype(int)

# Drop missing values where production wasn't recorded
df_long = df_long.dropna(subset=["Production_Value"])

print("Cleaned & Transformed Shape:", df_long.shape)
print(df_long.head())

Cleaned & Transformed Shape: (1121643, 9)
  Area Code (M49)         Area Item Code (CPC)                           Item  \
0            '004  Afghanistan          '01371              Almonds, in shell   
2            '004  Afghanistan          '01341                         Apples   
3            '004  Afghanistan          '01343                       Apricots   
4            '004  Afghanistan           '0115                         Barley   
5            '004  Afghanistan       '22249.01  Butter and ghee of sheep milk   

   Element Code     Element Unit  Year  Production_Value  
0          5510  Production    t  1961               0.0  
2          5510  Production    t  1961           15100.0  
3          5510  Production    t  1961           32000.0  
4          5510  Production    t  1961          378000.0  
5          5510  Production    t  1961            4104.0  


In [12]:
# Drop rows where production is exactly 0
df_long = df_long[df_long["Production_Value"] > 0]

In [13]:
df = df_long[df_long["Production_Value"] > 0]

In [14]:
df.head()

,Area Code (M49),Area,Item Code (CPC),Item,Element Code,Element,Unit,Year,Production_Value
2,'004,Afghanistan,'01341,Apples,5510,Production,t,1961,15100.0
3,'004,Afghanistan,'01343,Apricots,5510,Production,t,1961,32000.0
4,'004,Afghanistan,'0115,Barley,5510,Production,t,1961,378000.0
5,'004,Afghanistan,'22249.01,Butter and ghee of sheep milk,5510,Production,t,1961,4104.0
6,'004,Afghanistan,'22241.01,Butter of cow milk,5510,Production,t,1961,7000.0


In [15]:
output_filename = "cleaned_faostat.csv"
print(f"💾 Saving cleaned data to {output_filename}...")
df.to_csv(output_filename, index=False)
print("✅ Done! You don't need to run this script ever again.")

💾 Saving cleaned data to cleaned_faostat.csv...
✅ Done! You don't need to run this script ever again.


In [16]:


print(df.info())

print("\n--- 2. FIRST 5 ROWS ---")
display(df.head())

print("\n--- 3. UNIQUE COUNTRY NAMES IN YOUR DATA ---")
# Replace 'Country' with your actual country column name if it's named differently
country_column = "Area"

if country_column in df.columns:
    unique_countries = sorted(df[country_column].dropna().unique())
    print(f"Found {len(unique_countries)} unique countries:")
    print(unique_countries)
else:
    print(
        f"Could not find a column named '{country_column}'. Here are your columns: {list(df.columns)}"
    )

<class 'pandas.core.frame.DataFrame'>
Index: 1084619 entries, 2 to 1444274
Data columns (total 9 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Area Code (M49)   1084619 non-null  object 
 1   Area              1084619 non-null  object 
 2   Item Code (CPC)   1084619 non-null  object 
 3   Item              1084619 non-null  object 
 4   Element Code      1084619 non-null  int64  
 5   Element           1084619 non-null  object 
 6   Unit              1084619 non-null  object 
 7   Year              1084619 non-null  int64  
 8   Production_Value  1084619 non-null  float64
dtypes: float64(1), int64(2), object(6)
memory usage: 82.7+ MB
None

--- 2. FIRST 5 ROWS ---


,Area Code (M49),Area,Item Code (CPC),Item,Element Code,Element,Unit,Year,Production_Value
2,'004,Afghanistan,'01341,Apples,5510,Production,t,1961,15100.0
3,'004,Afghanistan,'01343,Apricots,5510,Production,t,1961,32000.0
4,'004,Afghanistan,'0115,Barley,5510,Production,t,1961,378000.0
5,'004,Afghanistan,'22249.01,Butter and ghee of sheep milk,5510,Production,t,1961,4104.0
6,'004,Afghanistan,'22241.01,Butter of cow milk,5510,Production,t,1961,7000.0



--- 3. UNIQUE COUNTRY NAMES IN YOUR DATA ---
Found 203 unique countries:
['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China, Hong Kong SAR', 'China, Macao SAR', 'China, Taiwan Province of', 'China, mainland', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czechia', "Côte d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Isl